In [2]:
# Credit Card Fraud Detection - Model Training

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Data Manipulation
import pandas as pd
import numpy as np

# Model Saving
import joblib

# Train-Test Split
from sklearn.model_selection import train_test_split

# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [3]:
df= pd.read_csv('/FraudGuard-AI/data/raw/creditcard.csv')

In [4]:
df.sample(5)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
62507,50296.0,1.196726,-0.074092,0.324433,0.193708,-0.217818,0.082010,-0.234457,0.157474,0.242674,...,-0.194716,-0.461364,0.045550,-0.273883,0.254545,0.479086,-0.033395,-0.007205,4.00,0
236271,148740.0,2.058133,-0.087814,-1.103497,0.396041,-0.111202,-1.097596,0.143414,-0.283550,0.601045,...,-0.292289,-0.718843,0.342691,-0.093951,-0.316605,0.206676,-0.070078,-0.062009,1.79,0
85667,60876.0,-0.305513,1.177800,1.085639,-0.071404,0.364781,-0.652305,0.767869,-0.090495,-0.494911,...,-0.287732,-0.714711,-0.087895,-0.179835,-0.057844,0.102128,0.250202,0.094898,4.99,0
23156,32617.0,-0.500427,0.291538,1.398809,-0.884914,-0.495028,-0.916347,0.863475,-0.085369,-0.044627,...,-0.045961,-0.435865,0.400819,0.378138,-0.989446,0.473639,0.041919,0.179521,108.87,0
10548,17444.0,-1.605223,-0.749286,1.755427,-0.747386,-1.697518,0.880305,1.676917,0.083032,1.427357,...,0.083441,-0.182780,0.954119,-0.055234,0.107699,0.797875,0.060574,0.168799,500.00,0


In [5]:
# Remove Duplicate Records

print("Duplicate Rows:", df.duplicated().sum())

df = df.drop_duplicates()

print("Shape After Removing Duplicates:", df.shape)

Duplicate Rows: 1081
Shape After Removing Duplicates: (283726, 31)


In [6]:
# Features and Targetabs
X = df.drop("Class", axis=1)

y = df["Class"]

print(X.shape)
print(y.shape)

(283726, 30)
(283726,)


In [7]:
# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (226980, 30)
Testing Shape: (56746, 30)


In [21]:
negative_count = y_train.value_counts()[0]
positive_count = y_train.value_counts()[1]

scale_pos_weight = negative_count / positive_count

print(f"Negative Samples : {negative_count}")
print(f"Positive Samples : {positive_count}")

print(f"scale_pos_weight : {scale_pos_weight:.2f}")

Negative Samples : 226602
Positive Samples : 378
scale_pos_weight : 599.48


In [22]:
# Create Machine Learning Models

models = {

    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        random_state=42,
        max_iter=1000
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight
    )

}

print(f"Total Models : {len(models)}")

Total Models : 4


In [23]:
# Model Evaluation Function

def evaluate_model(model, X_train, X_test, y_train, y_test):

    # Train Model
    model.fit(X_train, y_train)

    # Prediction
    y_pred = model.predict(X_test)

    # Probability Prediction
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    return {
        "Model": model,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc
    }

In [24]:
# Train All Models

results = []

trained_models = {}

for model_name, model in models.items():

    print("=" * 60)
    print(f"Training {model_name}")
    print("=" * 60)

    metrics = evaluate_model(
        model,
        X_train,
        X_test,
        y_train,
        y_test
    )

    metrics["Model Name"] = model_name

    trained_models[model_name] = metrics["Model"]

    results.append(metrics)

print("\nTraining Completed Successfully")

Training Logistic Regression
Training Decision Tree
Training Random Forest
Training XGBoost

Training Completed Successfully


In [25]:
# Compare Models

results_df = pd.DataFrame(results)

results_df = results_df.drop(columns=["Model"])

results_df = results_df[
    [
        "Model Name",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC AUC"
    ]
]

results_df.sort_values(
    by="F1 Score",
    ascending=False,
    inplace=True
)

results_df.reset_index(drop=True, inplace=True)

results_df

,Model Name,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,XGBoost,0.999595,0.973684,0.778947,0.865497,0.972770
1,Random Forest,0.999489,0.971429,0.715789,0.824242,0.944683
2,Decision Tree,0.999013,0.734940,0.642105,0.685393,0.820858
3,Logistic Regression,0.972192,0.050334,0.873684,0.095183,0.957940


In [28]:
best_model = trained_models["XGBoost"]

joblib.dump(best_model, "D:/FraudGuard-AI/models/fraud_model.pkl")
print("Model saved successfully!")

Model saved successfully!


In [29]:
loaded_model = joblib.load("D:/FraudGuard-AI/models/fraud_model.pkl")

print(loaded_model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)


In [30]:
sample = X_test.iloc[[0]]

print(model.predict(sample))
print(model.predict_proba(sample))

[0]
[[9.9999964e-01 3.4404894e-07]]
